# 03 · Medição mensal da qualidade do código

**Objetivo.** Medir, ao fim de cada mês entre 12 meses antes da adoção e setembro de 2026, os avisos de linter, a complexidade e os comentários de dívida técnica dos arquivos Python de cada projeto do piloto.

- **Lê:** `../out/piloto.csv` e os repositórios clonados em `../repos/<projeto>/` (via `git`); nas células de verificação, relê os `arquivos_mensais.parquet`
- **Grava:** `../out/<projeto>/arquivos_mensais.parquet` e `../out/<projeto>/avisos_por_blob.parquet` (um par por projeto)

Nos caminhos, `<projeto>` é o nome do repositório com `/` trocado por `__`.

**O que esta célula faz.** Define as funções de medição: para cada mês, pega o último commit do first-parent (a sequência que segue só o primeiro pai de cada merge, isto é, o histórico do branch principal) e lista os `.py` com o seu blob (objeto do Git que guarda o conteúdo de uma versão do arquivo). Cada blob distinto é lido e medido uma só vez: o Ruff conta avisos (regras F, E, B, SIM, C90, PL* e ERA, ignorando E501, PLR2004 e PLC0415), o Radon mede SLOC (linhas de código sem brancos nem comentários) e complexidade ciclomática por função, e conta-se SATD (self-admitted technical debt: comentários com TODO, FIXME, HACK ou XXX). O resultado é uma tabela (mês, arquivo, blob, medidas) por projeto; um projeto com mais de 5000 arquivos `.py` num mês seria excluído por custo.

In [1]:
import importlib.util, io, json, os, re, subprocess, sys, tempfile, tokenize
import pandas as pd
from radon.complexity import cc_visit
from radon.raw import analyze

RUFF_SELECT = 'F,E,B,SIM,C90,PLE,PLW,PLR,PLC,ERA'
RUFF_IGNORE = 'E501,PLR2004,PLC0415'
EXCL = re.compile(r'(^|/)(\.venv|venv|env|site-packages|node_modules|vendor|third_party|build|dist|\.tox)/')
SATD = re.compile(r'\b(TODO|FIXME|HACK|XXX)\b')
RUFF = [sys.executable, '-m', 'ruff'] if importlib.util.find_spec('ruff') else ['ruff']
LIMITE_ARQUIVOS = 5000          # acima disso o repositório é excluído por custo
FIM = pd.Timestamp('2026-10-01', tz='UTC')   # observa até setembro de 2026


def git(repo, *args, entrada=None):
    r = subprocess.run(['git', '-C', repo, *args], capture_output=True, input=entrada)
    if r.returncode != 0:
        raise RuntimeError(r.stderr.decode('utf-8', 'replace')[:300])
    return r.stdout


def meses_da_janela(adocao):
    """Fronteiras (1º dia de cada mês, UTC): de 12 meses antes da adoção até out/2026."""
    a = adocao.tz_convert('UTC')
    ini = pd.Timestamp(a.year, a.month, 1, tz='UTC') - pd.DateOffset(months=12)
    return list(pd.date_range(ini + pd.DateOffset(months=1), FIM, freq='MS', tz='UTC'))


def snapshots(repo, fronteiras):
    """4.1 e 4.2: para cada mês, o último commit do first-parent e os .py (caminho, blob)."""
    linhas = []
    for f in fronteiras:
        sha = git(repo, 'rev-list', '-1', '--first-parent', f'--before={f.strftime("%Y-%m-%dT%H:%M:%SZ")}', 'HEAD').decode().strip()
        if not sha:
            continue                       # ainda não havia commits
        mes = (f - pd.DateOffset(months=1)).strftime('%Y-%m')
        for l in git(repo, 'ls-tree', '-r', '-z', sha).decode('utf-8', 'replace').split('\0'):
            if not l:
                continue
            meta, caminho = l.split('\t', 1)
            tipo, blob = meta.split()[1:3]
            if tipo == 'blob' and caminho.endswith('.py') and not EXCL.search(caminho):
                linhas.append((mes, sha, caminho, blob))
    return pd.DataFrame(linhas, columns=['mes', 'sha', 'arquivo', 'blob'])


def ler_blobs(repo, blobs):
    """4.3: lê o conteúdo de cada blob distinto, uma só vez."""
    saida = git(repo, 'cat-file', '--batch', entrada=('\n'.join(blobs) + '\n').encode())
    conteudo, i = {}, 0
    while i < len(saida):
        fim = saida.index(b'\n', i)
        sha, _, tam = saida[i:fim].decode().split()
        tam = int(tam)
        conteudo[sha] = saida[fim + 1: fim + 1 + tam].decode('utf-8', 'replace')
        i = fim + 1 + tam + 1
    return conteudo


def medir_blobs(conteudo):
    """4.4: Ruff (avisos), Radon (complexidade, SLOC) e SATD de cada blob."""
    med, avisos = {}, []
    with tempfile.TemporaryDirectory() as tmp:
        for b, src in conteudo.items():
            with open(os.path.join(tmp, b + '.py'), 'w', encoding='utf-8') as f:
                f.write(src)
        r = subprocess.run([*RUFF, 'check', '--isolated', '--no-cache', '--exit-zero',
                            '--output-format', 'json', '--select', RUFF_SELECT, '--ignore', RUFF_IGNORE, tmp],
                           capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError('Ruff falhou: ' + r.stderr[:300])
        achados = json.loads(r.stdout or '[]')
    for a in achados:
        b = os.path.basename(a['filename'])[:-3]
        if a['code'] is None:                       # erro de sintaxe, não é aviso de qualidade
            med.setdefault(b, {})['erro_sintaxe'] = True
            continue
        linhas = conteudo[b].splitlines()
        row = a['location']['row']
        texto = linhas[row - 1].strip() if 0 < row <= len(linhas) else ''
        avisos.append((b, a['code'], row, texto))
    for b, src in conteudo.items():
        m = med.setdefault(b, {})
        try:
            blocos = cc_visit(src)
            cc = [x.complexity for x in blocos]
            m['sloc'] = analyze(src).sloc
            m['n_funcoes'] = len(cc)
            m['cc_media'] = sum(cc) / len(cc) if cc else 0.0
            m['cc_max'] = max(cc, default=0)
            m['cc_total'] = sum(cc)
        except Exception:
            m['erro_sintaxe'] = True
        satd = 0
        try:
            for t in tokenize.generate_tokens(io.StringIO(src).readline):
                if t.type == tokenize.COMMENT and SATD.search(t.string):
                    satd += 1
        except (tokenize.TokenError, IndentationError, SyntaxError):
            pass
        m['satd'] = satd
    df = pd.DataFrame.from_dict(med, orient='index').rename_axis('blob').reset_index()
    av = pd.DataFrame(avisos, columns=['blob', 'regra', 'linha', 'texto'])
    df['avisos'] = df.blob.map(av.blob.value_counts()).fillna(0).astype(int)
    compl = av[av.regra.str.match(r'C901|PLR09')]          # avisos de complexidade
    df['avisos_complexidade'] = df.blob.map(compl.blob.value_counts()).fillna(0).astype(int)
    return df, av


def medir_repo(nome, adocao, base='..'):
    chave = nome.replace('/', '__')
    repo, saida = f'{base}/repos/{chave}', f'{base}/out/{chave}'
    os.makedirs(saida, exist_ok=True)
    snap = snapshots(repo, meses_da_janela(pd.Timestamp(adocao)))
    if snap.groupby('mes').size().max() > LIMITE_ARQUIVOS:
        print(f'{nome}: mais de {LIMITE_ARQUIVOS} arquivos .py, excluído por custo'); return None
    conteudo = ler_blobs(repo, snap.blob.unique())
    med, avisos = medir_blobs(conteudo)
    tab = snap.merge(med, on='blob')
    tab.to_parquet(f'{saida}/arquivos_mensais.parquet')
    avisos.to_parquet(f'{saida}/avisos_por_blob.parquet')
    print(f'{nome}: {snap.mes.nunique()} meses, {len(snap)} linhas, {snap.blob.nunique()} versões medidas, '
          f'{len(avisos)} avisos')
    return tab

**O que esta célula faz.** Roda `medir_repo` para cada projeto de `piloto.csv`, usando a data do primeiro PR aceito como adoção, grava os dois arquivos `.parquet` do projeto e imprime um resumo (meses, linhas, versões medidas e avisos).

In [3]:
piloto = pd.read_csv('../out/piloto.csv')
for _, p in piloto.iterrows():
    medir_repo(p.full_name, p.primeiro)

numbagg/numbagg: 28 meses, 669 linhas, 141 versões medidas, 561 avisos


<unknown>:1475: SyntaxWarning: invalid escape sequence '\D'
<unknown>:1691: SyntaxWarning: invalid escape sequence '\D'
<unknown>:1714: SyntaxWarning: invalid escape sequence '\D'


econ-ark/HARK: 29 meses, 3667 linhas, 684 versões medidas, 9376 avisos
NewFuture/DDNS: 27 meses, 1519 linhas, 291 versões medidas, 600 avisos
lidofinance/diffyscan: 29 meses, 538 linhas, 159 versões medidas, 611 avisos
airbytehq/PyAirbyte: 33 meses, 5495 linhas, 1140 versões medidas, 1588 avisos


**O que isto mostra.** Os cinco projetos foram medidos (nenhum atingiu o limite de 5000 arquivos `.py` que causaria exclusão), com 27 meses em DDNS, 28 em numbagg, 29 em HARK e diffyscan e 33 em PyAirbyte. O número de linhas (arquivo, mês) supera o de versões medidas, por exemplo 3667 contra 684 em HARK e 5495 contra 1140 em PyAirbyte, o que indica que muitos arquivos permanecem idênticos entre meses e foram medidos uma só vez. Os avisos do Ruff nas versões medidas somam 561 (numbagg), 9376 (HARK), 600 (DDNS), 611 (diffyscan) e 1588 (PyAirbyte), contagens brutas, sem normalização pelo tamanho do código, em que HARK se destaca. As três mensagens de `SyntaxWarning` (invalid escape sequence, linhas 1475, 1691 e 1714) são avisos do Python sobre código-fonte analisado e não interromperam a medição.

**O que esta célula faz.** Silencia os `SyntaxWarning` que o Python emite ao analisar código-fonte com sequências de escape inválidas, como os que apareceram na saída da célula anterior. Não altera nenhuma medição.

In [4]:
import warnings
warnings.filterwarnings('ignore', category=SyntaxWarning)

**O que esta célula faz.** Verificação: para cada projeto, lê `arquivos_mensais.parquet` e imprime o número de arquivos `.py` no primeiro e no último mês da janela e quantos registros foram marcados com erro de sintaxe.

In [5]:
for _, p in piloto.iterrows():
    t = pd.read_parquet(f"../out/{p.full_name.replace('/', '__')}/arquivos_mensais.parquet")
    n = t.groupby('mes').size()
    erros = t['erro_sintaxe'].sum() if 'erro_sintaxe' in t else 0
    print(p.full_name, 'primeiro mês:', n.iloc[0], '| último mês:', n.iloc[-1], '| arquivos com erro de sintaxe:', erros)

numbagg/numbagg primeiro mês: 21 | último mês: 29 | arquivos com erro de sintaxe: 0
econ-ark/HARK primeiro mês: 110 | último mês: 151 | arquivos com erro de sintaxe: 4
NewFuture/DDNS primeiro mês: 15 | último mês: 108 | arquivos com erro de sintaxe: 0
lidofinance/diffyscan primeiro mês: 8 | último mês: 34 | arquivos com erro de sintaxe: 0
airbytehq/PyAirbyte primeiro mês: 61 | último mês: 261 | arquivos com erro de sintaxe: 0


**O que isto mostra.** Todos os projetos têm arquivos `.py` no primeiro mês da janela (numbagg 21, HARK 110, DDNS 15, diffyscan 8, PyAirbyte 61) e o número cresce até o último mês (29, 151, 108, 34 e 261), o que confirma que as tabelas mensais estão preenchidas nas duas pontas da janela. Apenas HARK tem erro de sintaxe (4 registros; 0 nos demais), o que limita a perda de medições por esse motivo. A contagem soma registros (arquivo, mês), não arquivos distintos.

**O que esta célula faz.** Lista os registros de HARK marcados com erro de sintaxe, com o mês e o caminho do arquivo, para identificar de onde vêm os 4 casos contados na célula anterior.

In [6]:
t = pd.read_parquet('../out/econ-ark__HARK/arquivos_mensais.parquet')
t[t.erro_sintaxe == True][['mes', 'arquivo']]

,mes,arquivo
1,2024-05,Documentation/example_notebooks/GenIncProcessM...
2,2024-05,Documentation/example_notebooks/JourneyPhDpara...
111,2024-06,Documentation/example_notebooks/GenIncProcessM...
112,2024-06,Documentation/example_notebooks/JourneyPhDpara...


**O que isto mostra.** Os 4 registros de HARK são de 2024-05 e 2024-06, com dois pares de caminhos que se repetem nos dois meses (aparentemente dois arquivos, com nomes truncados na saída, começando por `GenIncProcessM...` e `JourneyPhDpara...`), todos em `Documentation/example_notebooks`. Isso explica a contagem 4 da célula anterior (2 arquivos em 2 meses). A saída não permite dizer a causa dos erros de sintaxe.